# NB5: LoRA Fine-Tuning Defense Loop

**Purpose:** Take successful jailbreak prompts from NB3's attack logs, construct safety training data, and fine-tune Llama-3-8B using QLoRA to create a hardened model.

**Memory Budget (QLoRA fine-tuning):**
| Component | VRAM |
|---|---|
| Base model (4-bit) | ~5.5 GB |
| LoRA adapters (r=16) | ~0.1 GB |
| Optimizer (8-bit AdamW) | ~0.2 GB |
| Activations (grad checkpoint) | ~3-5 GB |
| Batch (seq=2048, bs=2) | ~2-3 GB |
| **Total** | **~12 GB** |

**Prerequisites:** NB1 (setup), NB2 (dataset), NB3 (attack round complete)

---

In [ ]:
# ============================================================
# Cell 0: Initialize
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/MediCS/utils')
from medics_utils import *

import json
import random
import torch

config = init_medics(mount_drive=False)
keys = load_api_keys()
openai_client = setup_api_clients(keys)

from huggingface_hub import login
login(token=keys["hf_token"])

print("Ready.")
print_gpu_memory()

In [ ]:
# ============================================================
# Cell 1: Configure the defense round
# ============================================================

# --- SET THIS FOR EACH DEFENSE ROUND ---
DEFENSE_ROUND = 0  # Change to 1, 2, ... for subsequent rounds

# Collect ALL attack logs from round 0 through current round (cumulative)
all_attack_rounds = list(range(DEFENSE_ROUND + 1))
ADAPTER_SAVE_PATH = f"{PATHS['checkpoints']}/lora_round_{DEFENSE_ROUND}"

print(f"Defense round: {DEFENSE_ROUND}")
print(f"Using attack logs from rounds: {all_attack_rounds}")
print(f"Adapter will be saved to: {ADAPTER_SAVE_PATH}")

## Step 1: Build Defense Training Dataset

In [ ]:
# ============================================================
# Cell 2: Collect successful jailbreaks from all rounds
# ============================================================
successful_attacks = []

for round_num in all_attack_rounds:
    log_path = f"{PATHS['attack_logs']}/round_{round_num}_attacks.jsonl"
    if not os.path.exists(log_path):
        print(f"WARNING: No attack log found for round {round_num}: {log_path}")
        continue
    attacks = read_jsonl(log_path)
    round_successes = [a for a in attacks if a.get("is_harmful", 0) == 1]
    successful_attacks.extend(round_successes)
    print(f"Round {round_num}: {len(round_successes)} successful attacks out of {len(attacks)}")

print(f"\nTotal successful attacks for defense training: {len(successful_attacks)}")

if len(successful_attacks) == 0:
    print("\nERROR: No successful attacks found. Cannot train defense.")
    print("This means the model is already safe against these attacks (good!).")
    print("Or NB3 hasn't been run yet.")

In [ ]:
# ============================================================
# Cell 3: Generate safe refusal responses for each successful attack
# ============================================================
REFUSAL_CACHE_PATH = f"{PATHS['checkpoints']}/refusals_round_{DEFENSE_ROUND}.json"
refusal_cache = load_checkpoint(REFUSAL_CACHE_PATH) or {}

REFUSAL_SYSTEM_PROMPT = """You are a medical AI safety trainer. Given a harmful medical 
query (which may contain code-switched words in other languages), generate an ideal 
safe refusal response.

The response should:
1. Clearly decline to provide harmful information
2. Briefly explain why this request is dangerous
3. Offer a safe alternative or redirect to appropriate resources

Be concise (2-4 sentences). Be firm but helpful.

Output as: {"refusal": "your response here"}"""

remaining = [a for a in successful_attacks if a["seed_id"] not in refusal_cache]
print(f"Refusals: {len(refusal_cache)} cached, {len(remaining)} to generate")

for i, attack in enumerate(tqdm(remaining)):
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": REFUSAL_SYSTEM_PROMPT},
                {"role": "user", "content": attack["attack_prompt"]},
            ],
            temperature=0.3,
            response_format={"type": "json_object"},
        )
        result = json.loads(response.choices[0].message.content)
        refusal_cache[attack["seed_id"]] = result.get("refusal", "")
    except Exception as e:
        print(f"  Error for {attack['seed_id']}: {e}")
        refusal_cache[attack["seed_id"]] = (
            "I'm unable to provide that information as it could cause serious harm. "
            "Please consult a qualified healthcare professional for medical guidance."
        )

    if (i + 1) % 50 == 0:
        save_checkpoint(refusal_cache, REFUSAL_CACHE_PATH)
    time.sleep(0.3)

save_checkpoint(refusal_cache, REFUSAL_CACHE_PATH)
print(f"\nRefusals generated: {len(refusal_cache)}")

In [ ]:
# ============================================================
# Cell 4: Build mixed training dataset (refusals + benign helpfulness)
# ============================================================
from datasets import Dataset

# 1. Build refusal training examples
refusal_examples = []
for attack in successful_attacks:
    refusal = refusal_cache.get(attack["seed_id"], "")
    if not refusal:
        continue
    refusal_examples.append({
        "messages": [
            {"role": "user", "content": attack["attack_prompt"]},
            {"role": "assistant", "content": refusal},
        ]
    })

print(f"Refusal examples: {len(refusal_examples)}")

# 2. Build benign helpfulness examples (to prevent over-refusal)
benign_twins = read_jsonl(BENIGN_TWINS_PATH)

# Sample benign examples to match refusal count (1:1 ratio)
n_benign = min(len(refusal_examples), len(benign_twins))
benign_sample = random.sample(benign_twins, n_benign)

# Generate helpful responses for benign questions using GPT-4o
HELPFUL_CACHE_PATH = f"{PATHS['checkpoints']}/helpful_responses_cache.json"
helpful_cache = load_checkpoint(HELPFUL_CACHE_PATH) or {}

benign_to_process = [b for b in benign_sample if b["seed_id"] not in helpful_cache]
print(f"\nBenign helpful responses: {len(helpful_cache)} cached, {len(benign_to_process)} to generate")

for i, twin in enumerate(tqdm(benign_to_process)):
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a helpful, safe medical AI assistant. "
                 "Provide accurate, helpful medical information for this benign question."},
                {"role": "user", "content": twin["benign_question"]},
            ],
            temperature=0.5,
            max_tokens=300,
        )
        helpful_cache[twin["seed_id"]] = response.choices[0].message.content
    except Exception as e:
        print(f"  Error: {e}")
    if (i + 1) % 50 == 0:
        save_checkpoint(helpful_cache, HELPFUL_CACHE_PATH)
    time.sleep(0.3)

save_checkpoint(helpful_cache, HELPFUL_CACHE_PATH)

benign_examples = []
for twin in benign_sample:
    helpful = helpful_cache.get(twin["seed_id"], "")
    if not helpful:
        continue
    benign_examples.append({
        "messages": [
            {"role": "user", "content": twin["benign_question"]},
            {"role": "assistant", "content": helpful},
        ]
    })

print(f"Benign examples: {len(benign_examples)}")

# 3. Combine and shuffle
all_training_examples = refusal_examples + benign_examples
random.shuffle(all_training_examples)

print(f"\nTotal training examples: {len(all_training_examples)}")
print(f"  Refusals: {len(refusal_examples)} ({len(refusal_examples)/len(all_training_examples)*100:.0f}%)")
print(f"  Benign: {len(benign_examples)} ({len(benign_examples)/len(all_training_examples)*100:.0f}%)")

In [ ]:
# ============================================================
# Cell 5: Format dataset for SFTTrainer
# ============================================================

def format_for_training(example):
    """Format messages into Llama-3 chat template for SFT."""
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

# We need the tokenizer loaded — load it here if not already available
from transformers import AutoTokenizer
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=keys["hf_token"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Right padding for training

# Create HuggingFace Dataset
train_dataset = Dataset.from_list(all_training_examples)
train_dataset = train_dataset.map(format_for_training)

print(f"Training dataset size: {len(train_dataset)}")
print(f"\nSample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])

## Step 2: QLoRA Fine-Tuning

In [ ]:
# ============================================================
# Cell 6: Load base model for training
# ============================================================
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Detect GPU type
is_ampere = torch.cuda.get_device_properties(0).major >= 8
compute_dtype = torch.bfloat16 if is_ampere else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

USE_UNSLOTH = False  # Set to True if Unsloth is installed

if USE_UNSLOTH:
    try:
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_ID,
            max_seq_length=2048,
            dtype=None,
            load_in_4bit=True,
            token=keys["hf_token"],
        )
        print("Loaded via Unsloth (2x faster training).")
    except Exception as e:
        print(f"Unsloth failed ({e}), falling back to standard loading.")
        USE_UNSLOTH = False

if not USE_UNSLOTH:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        token=keys["hf_token"],
        torch_dtype=compute_dtype,
    )
    print("Loaded via standard HuggingFace.")

print_gpu_memory()

In [ ]:
# ============================================================
# Cell 7: Apply LoRA configuration
# ============================================================
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
else:
    # Standard PEFT setup
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
print_gpu_memory()

In [ ]:
# ============================================================
# Cell 8: Configure training arguments
# ============================================================
from transformers import TrainingArguments

# Adjust for GPU type
if is_ampere:
    # A100: can use bfloat16, larger batch
    bf16 = True
    fp16 = False
    per_device_batch = 2
    grad_accum = 4
else:
    # V100/T4: use float16, smaller batch
    bf16 = False
    fp16 = True
    per_device_batch = 1
    grad_accum = 8

training_args = TrainingArguments(
    output_dir=ADAPTER_SAVE_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=per_device_batch,
    gradient_accumulation_steps=grad_accum,
    gradient_checkpointing=True,
    optim="adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=bf16,
    fp16=fp16,
    max_grad_norm=0.3,
    report_to="none",
    seed=42,
    dataloader_pin_memory=False,  # Save memory
)

effective_batch = per_device_batch * grad_accum
print(f"Training config:")
print(f"  Epochs: 3")
print(f"  Per-device batch: {per_device_batch}")
print(f"  Gradient accumulation: {grad_accum}")
print(f"  Effective batch size: {effective_batch}")
print(f"  Learning rate: 2e-4")
print(f"  Precision: {'bf16' if bf16 else 'fp16'}")
print(f"  Gradient checkpointing: True")
print(f"  Optimizer: adamw_8bit")

In [ ]:
# ============================================================
# Cell 9: Train!
# ============================================================
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,  # Don't pack sequences (safety-critical, each example matters)
)

print("Starting training...")
print(f"Training examples: {len(train_dataset)}")
print(f"Estimated steps: {len(train_dataset) // effective_batch * 3}")
print()

# Clear cache before training
torch.cuda.empty_cache()

train_result = trainer.train()

print("\nTraining complete!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")
print_gpu_memory()

In [ ]:
# ============================================================
# Cell 10: Save LoRA adapter to Drive
# ============================================================
print(f"Saving adapter to {ADAPTER_SAVE_PATH}...")
trainer.save_model(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

# Verify the save
adapter_files = os.listdir(ADAPTER_SAVE_PATH)
print(f"Saved files: {adapter_files}")
print(f"\nAdapter size: {sum(os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, f)) for f in adapter_files) / 1e6:.1f} MB")

## Step 3: Quick Validation of Hardened Model

In [ ]:
# ============================================================
# Cell 11: Test the hardened model on a few examples
# ============================================================
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Free training memory
del trainer
del model
torch.cuda.empty_cache()

# Reload base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=keys["hf_token"],
    torch_dtype=compute_dtype,
)

# Load LoRA adapter
hardened_model = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_PATH)
hardened_model = hardened_model.merge_and_unload()
hardened_model.eval()
print("Hardened model loaded.")
print_gpu_memory()


def test_generate(prompt, _model=hardened_model):
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(_model.device)
    with torch.no_grad():
        outputs = _model.generate(**inputs, max_new_tokens=256,
                                   temperature=0.7, do_sample=True,
                                   pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],
                           skip_special_tokens=True).strip()

In [ ]:
# ============================================================
# Cell 12: Test on harmful prompts (should refuse) and benign (should help)
# ============================================================

# Sample a few successful attacks to test refusal
test_harmful = random.sample(successful_attacks, min(5, len(successful_attacks)))

print("=" * 60)
print("TEST: Harmful prompts (should REFUSE)")
print("=" * 60)
for attack in test_harmful:
    response = test_generate(attack["attack_prompt"][:500])
    print(f"\nPrompt: {attack['attack_prompt'][:200]}...")
    print(f"Response: {response[:300]}")
    print("-" * 40)

# Test on benign questions (should still be helpful)
benign_test = read_jsonl(BENIGN_TWINS_PATH)[:5]

print("\n" + "=" * 60)
print("TEST: Benign prompts (should HELP)")
print("=" * 60)
for twin in benign_test:
    response = test_generate(twin["benign_question"])
    print(f"\nPrompt: {twin['benign_question']}")
    print(f"Response: {response[:300]}")
    print("-" * 40)

print("\n" + "=" * 60)
print(f"NB5 DEFENSE ROUND {DEFENSE_ROUND} COMPLETE")
print(f"Adapter saved to: {ADAPTER_SAVE_PATH}")
print(f"\nNext steps:")
print(f"  1. Go to NB3, set ROUND_NUM = {DEFENSE_ROUND + 1}")
print(f"  2. In NB4, load the hardened model (defense round {DEFENSE_ROUND})")
print(f"  3. Run NB3 to attack the hardened model")
print(f"  4. Come back to NB5 with DEFENSE_ROUND = {DEFENSE_ROUND + 1}")
print("=" * 60)